# 🚀 Upgraded Production Live Kaggle API (SmartSymptom AI)
### Multi-Threaded Flask + Ngrok + Top-3 Differential Diagnosis + Health Monitoring

Improvements in this notebook over the previous API:
1. **Multi-Threaded Flask Server**: `app.run(port=5000, threaded=True)` handles multiple concurrent requests without freezing.
2. **Top-3 Differential Diagnosis**: Returns primary illness + confidence score + top 3 candidates with percentages.
3. **Clinical Mapping**: Translates raw machine codes (`mel`, `nv`, `akiec`) into full clinical names.
4. **Health Check Endpoint**: `/health` reporting GPU status, memory allocated, and model uptime.
5. **Low-Confidence Warning**: Flags cases where the highest prediction has low confidence (< 45%).


In [ ]:
# ==========================================
# 1. INSTALLATION & SETUP
# ==========================================
!pip install -q pyngrok flask transformers torch tensorflow joblib Pillow gdown

import os
import io
import base64
import torch
import numpy as np
import tensorflow as tf
import joblib
from PIL import Image
from flask import Flask, request, jsonify
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Set ngrok auth token
NGROK_TOKEN = "3CWQc5zdydmULOprddOIFHcFX9m_7jdaa6moBtUxYAT8xD2Rg"
ngrok.set_auth_token(NGROK_TOKEN)

app = Flask(__name__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Server Compute Device: {device}")


In [ ]:
# ==========================================
# 2. DOWNLOAD & LOAD MODELS FROM HUGGING FACE
# ==========================================
from huggingface_hub import hf_hub_download

print("📥 Loading models directly from Hugging Face Hub...")

# --- A. Load Medical PubMedBERT Text Model ---
TEXT_REPO = "pathum0111/medical-text-pubmedbert"
try:
    print(f"Loading Text Model from {TEXT_REPO}...")
    text_model = AutoModelForSequenceClassification.from_pretrained(TEXT_REPO).to(device)
    tokenizer = AutoTokenizer.from_pretrained(TEXT_REPO)
    encoder_file = hf_hub_download(repo_id=TEXT_REPO, filename="label_encoder.pkl")
    le_text = joblib.load(encoder_file)
    print(f"✅ Text Model Loaded! Total Classes: {len(le_text.classes_)}")
except Exception as e:
    print(f"⚠️ Error loading text model from Hugging Face: {e}")
    text_model, tokenizer, le_text = None, None, None

# --- B. Load Skin Symptom Image Model ---
IMAGE_REPO = "pathum0111/symptom-image-classifier"
IMAGE_CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_MAPPING_IMAGE = {
    'akiec': 'Actinic Keratoses (Pre-cancerous lesion)',
    'bcc': 'Basal Cell Carcinoma (Skin cancer)',
    'bkl': 'Benign Keratosis (Seborrheic keratosis)',
    'df': 'Dermatofibroma (Benign skin nodule)',
    'mel': 'Melanoma (Malignant skin cancer - Urgent)',
    'nv': 'Melanocytic Nevus (Benign mole)',
    'vasc': 'Vascular Lesion (Angioma)'
}

image_model = None
# Try new EfficientNet first, then fallback to v2
for model_fname in ["best_skin_model_final.h5", "symptom_image_model_v2.h5"]:
    try:
        print(f"Attempting to load image model '{model_fname}' from {IMAGE_REPO}...")
        img_model_file = hf_hub_download(repo_id=IMAGE_REPO, filename=model_fname)
        image_model = tf.keras.models.load_model(img_model_file)
        print(f"✅ Image Model ({model_fname}) Loaded Successfully!")
        break
    except Exception as e:
        print(f"Notice: {model_fname} not found in {IMAGE_REPO}: {e}")


In [ ]:
# ==========================================
# 3. QUICK PRE-FLIGHT VERIFICATION
# ==========================================
print("")
print("🔍 --- RUNNING SELF-CHECK ---")
if text_model and tokenizer and le_text:
    sample_text = "I have sudden chest tightness, difficulty breathing, and high fever."
    inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        logits = text_model(**inputs).logits
        idx = torch.argmax(logits, dim=1).item()
    pred_disease = le_text.inverse_transform([idx])[0]
    print(f"✅ Text Model Test: '{sample_text}' -> PREDICTION: {pred_disease}")
else:
    print("⚠️ Text model not ready for self-check.")

if image_model:
    dummy_img = np.random.rand(1, 260, 260, 3).astype(np.float32)
    # Handle both (224, 224) and (260, 260) models
    in_shape = image_model.input_shape[1:3]
    if in_shape != (260, 260):
        dummy_img = np.random.rand(1, in_shape[0], in_shape[1], 3).astype(np.float32)
    preds = image_model.predict(dummy_img, verbose=0)
    top_c = IMAGE_CLASSES[preds.argmax()]
    print(f"✅ Image Model Test: Random image -> PREDICTION: {CLASS_MAPPING_IMAGE.get(top_c, top_c)}")
else:
    print("⚠️ Image model not ready for self-check.")
print("------------------------")


In [ ]:
# ==========================================
# 4. REST API ENDPOINTS
# ==========================================

@app.route('/health', methods=['GET'])
def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    gpu_mem = f"{torch.cuda.memory_allocated(0)/(1024**2):.1f} MB" if torch.cuda.is_available() else "N/A"
    return jsonify({
        "status": "healthy",
        "gpu": gpu_name,
        "gpu_memory": gpu_mem,
        "text_model_loaded": text_model is not None,
        "image_model_loaded": image_model is not None,
        "version": "2.0-production"
    }), 200

@app.route('/predict-text', methods=['POST'])
def predict_text():
    if text_model is None or tokenizer is None or le_text is None:
        return jsonify({"error": "Text model is not initialized", "status": "error"}), 500

    try:
        data = request.get_json(force=True)
        symptoms = data.get('symptoms', '').strip()
        top_k = int(data.get('top_k', 3))

        if not symptoms:
            return jsonify({"error": "Symptoms text cannot be empty", "status": "error"}), 400

        inputs = tokenizer(symptoms, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        text_model.eval()
        with torch.no_grad():
            logits = text_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

        top_indices = np.argsort(probs)[::-1][:top_k]
        candidates = []
        for idx in top_indices:
            disease_name = le_text.inverse_transform([idx])[0]
            conf = float(probs[idx])
            candidates.append({
                "illness": disease_name,
                "confidence": round(conf, 4),
                "percentage": f"{conf * 100:.1f}%"
            })

        primary = candidates[0]
        return jsonify({
            "status": "success",
            "illness": primary["illness"],
            "confidence": primary["confidence"],
            "is_low_confidence": primary["confidence"] < 0.40,
            "top_candidates": candidates
        }), 200

    except Exception as e:
        return jsonify({"error": str(e), "status": "error"}), 500

@app.route('/predict-image', methods=['POST'])
def predict_image():
    if image_model is None:
        return jsonify({"error": "Image model is not initialized", "status": "error"}), 500

    try:
        data = request.get_json(force=True)
        img_b64 = data.get('image', '').strip()
        top_k = int(data.get('top_k', 3))

        if not img_b64:
            return jsonify({"error": "Base64 image cannot be empty", "status": "error"}), 400

        # Strip data URL prefix if present
        if "base64," in img_b64:
            img_b64 = img_b64.split("base64,")[1]

        img_bytes = base64.b64decode(img_b64)
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        
        # Match model input shape
        input_shape = image_model.input_shape[1:3] # e.g. (224, 224) or (260, 260)
        img_resized = img.resize(input_shape)
        img_arr = np.array(img_resized) / 255.0
        img_arr = np.expand_dims(img_arr, axis=0)

        probs = image_model.predict(img_arr, verbose=0)[0]
        top_indices = np.argsort(probs)[::-1][:top_k]

        candidates = []
        for idx in top_indices:
            code = IMAGE_CLASSES[idx]
            conf = float(probs[idx])
            candidates.append({
                "code": code,
                "name": CLASS_MAPPING_IMAGE.get(code, code),
                "confidence": round(conf, 4),
                "percentage": f"{conf * 100:.1f}%"
            })

        primary = candidates[0]
        return jsonify({
            "status": "success",
            "condition": primary["code"],
            "name": primary["name"],
            "confidence": primary["confidence"],
            "is_low_confidence": primary["confidence"] < 0.40,
            "top_candidates": candidates
        }), 200

    except Exception as e:
        return jsonify({"error": str(e), "status": "error"}), 500


In [ ]:
# ==========================================
# 5. START MULTI-THREADED SERVER & NGROK TUNNEL
# ==========================================
# Kill any existing ngrok tunnels
ngrok.kill()

public_url = ngrok.connect(5000)
print("=" * 60)
print(f"🌍 LIVE PUBLIC API URL: {public_url}")
print(f"🩺 Health Check: {public_url}/health")
print("=" * 60)
print("🚀 Server running in MULTI-THREADED mode...")

# Run threaded Flask to handle concurrent requests without freezing
app.run(port=5000, threaded=True)
